# 59 - Checkmate Framework Diagnostic

**Purpose:** Analyze why the original Checkmate Framework backtest failed catastrophically.

**Key Findings:**
- Strategy: -37.5% return vs HODL: +30,253%
- Root cause: Position sizing went to 0% or negative during bull markets
- Signal was CORRECT (identified tops/bottoms), but response was wrong
- Bitcoin spent ~30% of time in 'overheated' zones where strategy exited

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
DATA_DIR = Path.home() / "Documents" / "bitcoin-lab-btc-data-pipeline" / "data" / "daily"

def load_metric(name):
    path = DATA_DIR / f"{name}.parquet"
    if not path.exists(): return pd.DataFrame(columns=['time', 'value'])
    df = pd.read_parquet(path)
    if 'time' in df.columns:
        df['time'] = pd.to_datetime(df['time'])
        if df['time'].dt.tz is not None: df['time'] = df['time'].dt.tz_localize(None)
        df = df.set_index('time').sort_index()
    return df

price = load_metric('price')
mvrv = load_metric('mvrv')
print(f"Loaded data: {price.index[0].date()} to {price.index[-1].date()}")

In [ ]:
# Analyze time spent in different MVRV zones
df = price[['value']].rename(columns={'value': 'price'})
df = df.join(mvrv[['value']].rename(columns={'value': 'mvrv'}))
df['mvrv'] = df['mvrv'].ffill()

# Original framework thresholds
zones = [
    ('Deep Value (<1.0)', df['mvrv'] < 1.0, '100%'),
    ('Value (1.0-1.5)', (df['mvrv'] >= 1.0) & (df['mvrv'] < 1.5), '75%'),
    ('Neutral (1.5-2.0)', (df['mvrv'] >= 1.5) & (df['mvrv'] < 2.0), '50%'),
    ('Elevated (2.0-2.4)', (df['mvrv'] >= 2.0) & (df['mvrv'] < 2.4), '0%'),
    ('Hot (2.4-3.0)', (df['mvrv'] >= 2.4) & (df['mvrv'] < 3.0), '-25%'),
    ('Euphoria (>3.0)', df['mvrv'] >= 3.0, '-50%'),
]

print("TIME SPENT IN EACH MVRV ZONE")
print("="*60)
print(f"{'Zone':<25} {'Time':>10} {'Original Pos':>15}")
print("-"*60)

for name, mask, pos in zones:
    pct = mask.sum() / len(df) * 100
    print(f"{name:<25} {pct:>9.1f}% {pos:>15}")

# Key insight
zero_or_short = (df['mvrv'] >= 2.0).sum() / len(df) * 100
print(f"\n⚠️ Strategy at 0% or SHORT: {zero_or_short:.1f}% of the time!")
print("   This is why it failed - missed entire bull markets!")

In [ ]:
# Returns analysis by MVRV zone
df['returns'] = df['price'].pct_change()
df['fwd_90d'] = df['price'].shift(-90) / df['price'] - 1

print("\nRETURNS BY MVRV ZONE")
print("="*60)
print(f"{'Zone':<25} {'Avg Daily':>12} {'Avg 90d Fwd':>15}")
print("-"*60)

for name, mask, _ in zones:
    daily = df.loc[mask, 'returns'].mean() * 365 * 100
    fwd = df.loc[mask, 'fwd_90d'].mean() * 100
    print(f"{name:<25} {daily:>+11.1f}% {fwd:>+14.1f}%")

print("\n💡 Key insight: Even 'overheated' zones have positive returns!")
print("   The market can stay overvalued longer than you can stay in cash.")

## Conclusion

**The signal was CORRECT but the position sizing response was WRONG:**

| Problem | Solution |
|---------|----------|
| Goes to 0% when MVRV > 2.0 | Keep minimum floor (20-30%) |
| Goes SHORT when MVRV > 2.4 | Never short Bitcoin |
| Binary response to signal | Gradual, asymmetric response |

**Next:** See notebook 60 for fixed version.